# Alignment of LLMs with Human Feedback (RLHF)

## 1. The Need for Alignment

While pre-training teaches a Large Language Model (LLM) the statistical structure of language (e.g., predicting the next word) and instruction fine-tuning teaches it to respond to specific tasks (e.g., summarization, translation), neither guarantees that the response aligns with human preferences.

An instruction-fine-tuned model may generate technically valid but practically unhelpful, unsafe, or non-human-like responses. For example:

* **Prompt:** "What's the best way to lose weight quickly?"
* **Technically valid but unsafe output:** "Stop eating entirely for a few days."
* **Aligned/Preferred output:** "Reduce carb intake, increase fiber and protein, and exercise regularly."

Alignment ensures outputs are **unbiased, fair, non-toxic, and human-like** by incorporating human preferences directly into the training loop.

---

## 2. Taxonomy of Alignment Methods

Alignment techniques are broadly categorized based on their optimization objectives and execution style (online vs. offline).

| Categorization | Explanation | Examples |
| --- | --- | --- |
| **Online Alignment** | The model generates outputs, receives feedback, and updates its parameters iteratively in a closed loop. | PPO (Proximal Policy Optimization) |
| **Offline Alignment** | The model is optimized using a pre-computed, static dataset of preferences without continuous generation/feedback loops. | DPO (Direct Preference Optimization) |
| **Reward Maximization** | Explicitly trains a separate "Reward Model" to score outputs, then optimizes the LLM to maximize this score. | RLHF via PPO |
| **Contrastive Learning** | Skips the explicit reward model and directly optimizes the LLM parameters based on pairwise preferences. | DPO |

*(Note: The transcript mentions "brain and DPG" under distribution matching, which likely refers to advanced alignment variants like distributional preference learning; ensure you know the broad categorization for MCQs).*

---

## 3. Reinforcement Learning (RL) Framework for LLMs

To apply RL to language models, we map traditional RL concepts (like a robot navigating a maze) to the text generation process.

* **Policy ($\pi_\theta$):** The LLM being fine-tuned. It is a probability distribution over the vocabulary (action space).
* **Action ($A_t$):** The selection of the *next single token* from the vocabulary $\vert{}V\vert{}$.
* **State ($S_t$):** The initial prompt *plus* all previously generated output tokens up to time $t$.
* **Environment:** The sequence generation process itself (there is no physical environment).
* **Reward ($R$):** The scalar score indicating output quality. **Crucial NPTEL Exception:** Unlike standard RL where rewards are received step-by-step, in LLM generation, intermediate token rewards are exactly $0$. The reward is only provided when the model generates the **End of Sentence (EOS)** token.

**Example of State-Action Transition:**

* **Prompt (Initial State $S_0$):** "What is the capital of India?"
* **Action $A_0$:** Model generates token `"The"` $\rightarrow$ transitions to $S_1$.
* **State $S_1$:** "What is the capital of India? The"
* **Action $A_1$:** Model generates `"capital"` $\rightarrow$ transitions to $S_2$.
* *Reward remains 0 until the final token (EOS) is reached.*

---

## 4. The Reward Model (RM)

The goal of the Reward Model is to act as a proxy for human judgment, providing a scalar score for any generated sequence to guide the Policy Model.

### Why use an AI Reward Model instead of Humans?

1. **Cost and Speed:** Human feedback is exceptionally expensive and slow.
2. **Scalability:** RL requires continuous feedback during millions of iterations.
To solve this, we train a separate LLM (the Reward Model) to mimic human preference.

### Training the Reward Model

Instead of asking humans to assign absolute scores (which is highly subjective, e.g., rating a poem 0.7 vs 0.8), we use **pairwise comparisons**.

* **Training Data Structure:** A prompt $x$, a preferred response $y^+$, and a non-preferred response $y^-$.
* **RLHF vs. RLAIF:** If humans annotate which response is better, it is Reinforcement Learning from Human Feedback (RLHF). If another AI (like GPT-4 or Anthropic's Constitutional AI principles) annotates the preferences, it is Reinforcement Learning from AI Feedback (RLLAIF).

### The Mathematical Formulation: Bradley-Terry (BT) Model

The Reward Model's cost function is derived from the **Bradley-Terry preference model**, which calculates the probability that entity $y_1$ is preferred over $y_2$:

$$P(y_1 \succ y_2) = \frac{p_1}{p_1 + p_2}$$

Using exponential functions to ensure positive values, where $r_i$ is the reward score:

$$P(y_1 \succ y_2) = \frac{\exp(r_1)}{\exp(r_1) + \exp(r_2)}$$

We parameterize this reward function using our Reward Model parameters ($\theta$). Given prompt $x$, preferred output $y^+$, and rejected output $y^-$:

$$P(y^+ \succ y^- \vert{} x) = \frac{\exp(r_\theta(x, y^+))}{\exp(r_\theta(x, y^+)) + \exp(r_\theta(x, y^-))}$$

### The Reward Model Objective Function

To train the RM, we maximize the log-likelihood of the human preferences. Taking the logarithm of the BT equation and simplifying (dividing numerator and denominator by $\exp(r_\theta(x, y^+))$) yields the log-sigmoid of the reward difference:

$$\text{Maximize } \log \sigma \left( r_\theta(x, y^+) - r_\theta(x, y^-) \right)$$

*Note: $\sigma$ represents the sigmoid function.* Through standard gradient descent, the RM learns to assign a higher scalar value to $y^+$ than to $y^-$.

---

## 5. The Policy Optimization Objective

Once the Reward Model $r_\theta(x, y)$ is trained, we use it to update the Policy Model $\pi_\theta$.

### Objective 1: Maximize Expected Reward

Because language generation is stochastic (one prompt can yield millions of possible outputs), we optimize for the **expected** reward over all generations:


$$\text{Maximize } \mathbb{E}_{y \sim \pi_\theta(y\vert{}x)}[r_\theta(x,y)]$$

### Objective 2: Prevent Reward Hacking (KL Divergence)

If we strictly maximize the reward, the Policy Model will "hack" the RM. It might learn that long paragraphs, excessive emojis, or overly polite buzzwords yield high scores, causing it to generate text that completely loses human readability or correctness.

To prevent this, we anchor the model to its initial instruction-tuned state using a **Reference Model** ($\pi_{ref}$). We penalize the Policy Model if its distribution diverges too far from the Reference Model. We measure this divergence using **Kullback-Leibler (KL) Divergence**:

$$D_{KL}(\pi_\theta(y\vert{}x) \vert{}\vert{} \pi_{ref}(y\vert{}x)) = \mathbb{E} \left[ \log \frac{\pi_\theta(y\vert{}x)}{\pi_{ref}(y\vert{}x)} \right]$$

### The Final RLHF Objective Function

We combine both objectives using a scaling factor $\lambda$ to balance reward maximization and the KL divergence penalty:

$$\text{Maximize } \mathbb{E}_{x \sim D, y \sim \pi_\theta} \left[ r_\theta(x,y) - \lambda \log \left( \frac{\pi_\theta(y\vert{}x)}{\pi_{ref}(y\vert{}x)} \right) \right]$$

---

## 6. NPTEL Practice Questions

**Q1. In the context of applying Reinforcement Learning to LLMs, when does the model typically receive a non-zero reward?**
a) After generating every single token.
b) Only after generating the End of Sentence (EOS) token.
c) At the beginning of the prompt sequence.
d) When the state transitions from $S_0$ to $S_1$.
**Answer:** b) Only after generating the End of Sentence (EOS) token.
*Explanation:* Intermediate tokens receive a reward of 0 because the coherence and preference of a response can only be evaluated once the entire sentence/sequence is complete.

**Q2. Why is KL divergence used in the final RLHF objective function?**
a) To increase the vocabulary size of the policy model.
b) To train the Bradley-Terry preference model.
c) To prevent the policy model from "reward hacking" and diverging too far from the initial reference model.
d) To convert offline alignment into online alignment.
**Answer:** c) To prevent the policy model from "reward hacking" and diverging too far from the initial reference model.
*Explanation:* Without KL divergence, the policy model optimizes solely for the reward score, often resulting in unnatural, verbose, or hacked outputs that exploit flaws in the reward model rather than generating high-quality text.

**Q3. If the reward model assigns a score of $R_1$ to output $Y_1$ and $R_2$ to output $Y_2$ for a given prompt, what does the Bradley-Terry model dictate is the probability that $Y_1$ is preferred over $Y_2$?**
a) $\log(R_1) - \log(R_2)$
b) $\sigma(R_1 \times R_2)$
c) $\frac{\exp(R_1)}{\exp(R_1) + \exp(R_2)}$
d) $\frac{R_1}{R_1 + R_2}$
**Answer:** c) $\frac{\exp(R_1)}{\exp(R_1) + \exp(R_2)}$
*Explanation:* The Bradley-Terry model calculates preference probability as the exponentiated reward of the preferred item divided by the sum of the exponentiated rewards of both items.

---

## 7. Quick Revision Summary

* **Goal:** Align instruction-tuned LLMs with human values (helpful, harmless, honest).
* **RL Map:** Policy = LLM; Action = Next token; State = Prompt + past tokens; Reward = Score at sequence end.
* **Reward Model (RM):** Trained via pairwise comparisons ($y^+$ vs $y^-$) to output a scalar quality score. Eliminates the need for humans at every inference step.
* **Bradley-Terry Model:** Mathematical foundation for training the RM, optimized using the log-sigmoid of the reward difference.
* **RLAIF (Constitutional AI):** Uses another AI (instead of humans) to generate preference data based on predefined ethical rules.
* **PPO Objective:** Maximize expected reward *minus* the KL divergence from a reference model (prevents reward hacking).